In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.local_votacao_secao")
display(df_bronze)

In [0]:
# Padronizando os nomes das colunas
df_local_votacao = df_bronze \
    .withColumnRenamed("local_votacao", "polling_place") \
    .withColumnRenamed("bairro", "neighborhood") \
    .withColumnRenamed("endereco", "address") \
    .withColumnRenamed("secao", "section_code") \
    .withColumnRenamed("zona", "zone_code") \

print(f"Total registros: {df_local_votacao.count()}")
display(df_local_votacao)


In [0]:
# Transformar/Converter colunas para o padarão para depois salvar a Silver V2

# from pyspark.sql.functions import col, trim, to_date, expr, when

# df_silver = (
#     df_eleicoes
#     .withColumn("election_date", to_date(col("election_date"), "M/d/yyyy"))
#     )

In [0]:
# criando o campo source_file
from pyspark.sql.functions import lit

df = df_local_votacao.withColumn("source_file", lit("local_votacao.csv"))

In [0]:
# Separando registros inválidos para quarentena

df_invalid = df.filter("""
zone_code IS NULL
OR section_code IS NULL
""")

print(f"Total registros inválidos: {df_invalid.count()}")
display(df_invalid)

In [0]:
# Salvando quarentena da Silver V2

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.local_votacao_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df_silver = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Filtrando valores válidos
from pyspark.sql.functions import col

df_silver_filtered = (
    df_silver.dropDuplicates()
      .filter(
          (col("polling_place").isNotNull()) &
          (col("section_code").isNotNull()) &
          (col("zone_code").isNotNull()) 
      )
)

display(df_silver_filtered)

In [0]:
# Criando o campo silver_processed_timestamp em df_silver_v2
from pyspark.sql.functions import current_timestamp

df_local_votacao = df_silver_filtered \
.withColumn("silver_processed_timestamp", current_timestamp()) \
.withColumn("created_at", current_timestamp()) \
.withColumn("updated_at", current_timestamp())

In [0]:
# Grantindo a deduplicação por chave de negócio

window_spec = Window.partitionBy(
    "ingestion_timestamp",
    "polling_place",
    "section_code",
).orderBy(
    col("ingestion_timestamp").desc()
)

df_local_votacao = (
    df_local_votacao
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# Validando shema da dataframe v2
df_local_votacao.printSchema()

In [0]:
# Salvando a tabela Silver em uma staging table

df_eleicoes.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_silver.local_votacao_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.local_votacao
AS
SELECT * FROM workspace.drs_silver.local_votacao_staging WHERE 1 = 0
""")

In [0]:
# Validar schema da staging

spark.table("workspace.drs_silver.local_votacao_staging").printSchema()

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.local_votacao AS target
USING workspace.drs_silver.local_votacao_staging AS source

ON target.zone_code = source.zone_code
AND target.section_code = source.section_code

WHEN MATCHED THEN
  UPDATE SET
    target.polling_place = source.polling_place,
    target.neighborhood = source.neighborhood,
    target.address = source.address,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_file = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (
    polling_place,
    neighborhood,
    address,
    section_code,
    zone_code,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
  )
  VALUES (
    source.polling_place,
    source.neighborhood,
    source.address,
    source.section_code,
    source.zone_code,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
  )
""")


In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
* 
--COUNT(*) AS total_rows
FROM workspace.drs_silver.local_votacao
""").display()

In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    ingestion_timestamp,
    polling_place,
    section_code,
    COUNT(*) AS qtd
FROM workspace.drs_silver.local_votacao
GROUP BY 1,2,3
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Data Quality checks da Silver V2
dq = spark.sql("""
SELECT
    SUM(CASE WHEN zone_code < 0 THEN 1 ELSE 0 END) AS vote_count,
    SUM(CASE WHEN section_code IS NULL THEN 1 ELSE 0 END) AS generation_date
    FROM workspace.drs_silver.local_votacao
""")

display(dq)

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.local_votacao_quarantine
""").display()

In [0]:
# # Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]

if (
    dq_row["vote_count"] > 0 or
    dq_row["generation_date"] > 0 
):
    raise Exception("Data Quality check failed in workspace.drs_silver.local_votacao")



